In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('breast_cancer.csv')
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# Data preprocessing
df = df.drop('id', axis=1)  # Remove ID column
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})  # Convert to binary

# Prepare features and target
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# Train-test split (80-20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")
print(f"Features: {X_train_scaled.shape[1]}")
print(f"Class distribution - Train: {y_train.value_counts().to_dict()}")
print(f"Class distribution - Test: {y_test.value_counts().to_dict()}")


Dataset shape: (569, 33)

First few rows:
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_wors

In [2]:
# Clean the dataset - remove the problematic Unnamed:32 column
df = df.drop('Unnamed: 32', axis=1, errors='ignore')
print("Cleaned dataset shape:", df.shape)
print("\nMissing values check:")
print(df.isnull().sum().sum())  # Should be 0 now
print("\nData types:")
print(df.dtypes.value_counts())

# Verify features are all numeric
feature_cols = df.drop('diagnosis', axis=1).columns
print(f"\nAll {len(feature_cols)} features are numeric: {df[feature_cols].apply(lambda x: pd.api.types.is_numeric_dtype(x)).all()}")

# Re-split after cleaning (to be consistent)
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (re-apply scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Clean data ready!")
print(f"Training samples: {X_train_scaled.shape[0]}")
print(f"Test samples: {X_test_scaled.shape[0]}")
print(f"Features: {X_train_scaled.shape[1]}")


Cleaned dataset shape: (569, 31)

Missing values check:
0

Data types:
float64    30
int64       1
Name: count, dtype: int64

All 30 features are numeric: True

✅ Clean data ready!
Training samples: 455
Test samples: 114
Features: 30


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Initialize baseline models
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Dictionary to store results
results = {}

print("=== BASELINE MODELS TRAINING ===\n")

for name, model in models.items():
    print(f"Training {name}...")

    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    results[name] = {'CV_Accuracy': cv_scores.mean(), 'CV_Std': cv_scores.std()}

    # Train on full training set
    model.fit(X_train_scaled, y_train)

    # Test predictions
    y_pred = model.predict(X_test_scaled)
    test_accuracy = (y_pred == y_test).mean()
    test_roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])

    results[name]['Test_Accuracy'] = test_accuracy
    results[name]['Test_ROC_AUC'] = test_roc_auc

    print(f"{name}:")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Test ROC-AUC: {test_roc_auc:.4f}")
    print(f"  Classification Report:\n{classification_report(y_test, y_pred, target_names=['Benign', 'Malignant'])}")
    print("-" * 50)

print("\n✅ Baseline models completed successfully!")
print("\nBaseline Results Summary:")
for name in results:
    print(f"{name}: Test Acc={results[name]['Test_Accuracy']:.4f}, ROC-AUC={results[name]['Test_ROC_AUC']:.4f}")



=== BASELINE MODELS TRAINING ===

Training Decision Tree...
Decision Tree:
  CV Accuracy: 0.9077 (±0.0397)
  Test Accuracy: 0.9211
  Test ROC-AUC: 0.9448
  Classification Report:
              precision    recall  f1-score   support

      Benign       0.91      0.97      0.94        72
   Malignant       0.95      0.83      0.89        42

    accuracy                           0.92       114
   macro avg       0.93      0.90      0.91       114
weighted avg       0.92      0.92      0.92       114

--------------------------------------------------
Training Logistic Regression...
Logistic Regression:
  CV Accuracy: 0.9736 (±0.0149)
  Test Accuracy: 0.9649
  Test ROC-AUC: 0.9960
  Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg  

In [4]:
import xgboost as xgb
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

print("=== HYBRID XGBoost + LightGBM TRAINING ===\n")

# Initialize base learners
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

lgbm_model = LGBMClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    verbose=-1  # Suppress LightGBM warnings
)

# Generate out-of-fold predictions for stacking (using same CV folds)
oof_xgb = np.zeros(len(X_train_scaled))
oof_lgbm = np.zeros(len(X_train_scaled))

print("Generating OOF predictions...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_scaled, y_train)):
    print(f"  Fold {fold+1}/5")

    X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # XGBoost OOF prediction
    xgb_model.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]

    # LightGBM OOF prediction
    lgbm_model.fit(X_tr, y_tr)
    oof_lgbm[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]

# Train meta-learner (Logistic Regression) on OOF predictions
meta_features = np.column_stack([oof_xgb, oof_lgbm])
meta_model = LogisticRegression(random_state=42, max_iter=1000)
meta_model.fit(meta_features, y_train)

# Test predictions for hybrid
print("\nGenerating test predictions...")
xgb_test_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]
lgbm_test_proba = lgbm_model.predict_proba(X_test_scaled)[:, 1]
hybrid_test_features = np.column_stack([xgb_test_proba, lgbm_test_proba])

hybrid_pred = meta_model.predict(hybrid_test_features)
hybrid_proba = meta_model.predict_proba(hybrid_test_features)[:, 1]

# Evaluate hybrid
hybrid_cv_accuracy = cross_val_score(meta_model, meta_features, y_train, cv=cv, scoring='accuracy').mean()
hybrid_test_accuracy = (hybrid_pred == y_test).mean()
hybrid_test_roc_auc = roc_auc_score(y_test, hybrid_proba)

results['XGBoost+LightGBM Hybrid'] = {
    'CV_Accuracy': hybrid_cv_accuracy,
    'Test_Accuracy': hybrid_test_accuracy,
    'Test_ROC_AUC': hybrid_test_roc_auc
}

print(f"\n✅ HYBRID MODEL RESULTS:")
print(f"  CV Accuracy: {hybrid_cv_accuracy:.4f}")
print(f"  Test Accuracy: {hybrid_test_accuracy:.4f}")
print(f"  Test ROC-AUC: {hybrid_test_roc_auc:.4f}")
print(f"\nHybrid Classification Report:")
print(classification_report(y_test, hybrid_pred, target_names=['Benign', 'Malignant']))

print("\n" + "="*60)


=== HYBRID XGBoost + LightGBM TRAINING ===

Generating OOF predictions...
  Fold 1/5
  Fold 2/5
  Fold 3/5
  Fold 4/5
  Fold 5/5

Generating test predictions...

✅ HYBRID MODEL RESULTS:
  CV Accuracy: 0.9714
  Test Accuracy: 0.9737
  Test ROC-AUC: 0.9904

Hybrid Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114




In [5]:
import joblib

# save scaler and models
joblib.dump(scaler, "scaler.pkl")
joblib.dump(xgb_model, "xgb_model.pkl")
joblib.dump(lgbm_model, "lgbm_model.pkl")
joblib.dump(meta_model, "meta_model.pkl")

print("Models saved successfully!")

Models saved successfully!
